### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or arguement definitions (often a JSON schema)
2. A function or coroutine to execute

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='<think>\nOkay, so the user is asking why parrots talk. Let me start by recalling what I know about parrots. I remember that they\'re part of the Psittaciformes order and are known for their ability to mimic sounds, including human speech. But why do they do that?\n\nFirst, maybe it\'s related to their natural behavior. In the wild, parrots often live in social groups. Mimicry could be a way they communicate with each other. For example, they might use calls to maintain group cohesion or signal danger. So, talking in the wild might help them stay in touch with their flock.\n\nThen, when they\'re kept as pets, they might talk to interact with their human caregivers. They could be trying to fit into their new social environment, much like they would with other parrots. If a parrot hears words often, it might repeat them to bond with the owner or get attention. Positive reinforcement, like when owners react to the parrot\'s speech, probably encourages this behavior.\n\nA

In [4]:
from langchain.tools import tool

## Now this tool library that we are importing here is used as a decorator (@tool) on top of the any function.
## That function will become tool in a langchain

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"

## Now we need to bind the tool with the model
model_with_tools = model.bind_tools([get_weather])

In [5]:
## Now we to call this (above one)

response = model_with_tools.invoke("What's the weather like in Boston?")
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

Tool: get_weather
Args: {'location': 'Boston'}


### Tool Execution Loop


In [8]:
# Step 1: Model generates tool calls
messages = [{"role":"user","content":"What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguements
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72F and sunny."

The weather in Boston is sunny.


In [9]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. I need to use the get_weather function. The function requires a location parameter. Boston is the location here. So I should call get_weather with location set to "Boston". Let me make sure there\'s no typo. Everything looks good. I\'ll format the tool call as specified.\n', 'tool_calls': [{'id': 'ryr7rp14h', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 153, 'total_tokens': 246, 'completion_time': 0.157716805, 'completion_tokens_details': {'reasoning_tokens': 69}, 'prompt_time': 0.006708378, 'prompt_tokens_details': None, 'queue_time': 0.051872802, 'total_time': 0.164425183}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_